In [3]:
from tqdm.notebook import tqdm
import pandas as pd

import sys
sys.path.append('..')
from src.web_scrapping_retail import WebScrappingRetail

In [4]:
## DataFrame con los datos de scrapping
scraper = WebScrappingRetail(max_por_categoria=100)
productos = scraper.run()

  678 subcategorías encontradas


Scraping categorías:  36%|███▌      | 244/678 [18:15<46:16,  6.40s/it]  

  Error en 'Cuadernos Y Agendas': HTTPSConnectionPool(host='www.jumbocolombia.com', port=443): Read timed out.


Scraping categorías:  59%|█████▉    | 400/678 [29:21<21:09,  4.57s/it]  

  Rate limit en 'Multiusos' — esperando 15s...


Scraping categorías:  59%|█████▉    | 400/678 [29:37<21:09,  4.57s/it]

  Rate limit en 'Multiusos' — esperando 15s...


Scraping categorías: 100%|██████████| 678/678 [51:16<00:00,  4.54s/it]  


  Total productos extraídos: 28,449


In [5]:
# Construir DataFrame
print(f'\n{"="*50}')
print(f'Total antes de limpiar: {len(productos):,}')

if len(productos) == 0:
    print('No se extrajo ningún producto.')
else:
    df = pd.DataFrame(productos)
    df = df.drop_duplicates(subset=['nombre', 'precio']).reset_index(drop=True)
    df['precio'] = pd.to_numeric(df['precio'], errors='coerce')
    df = df[['nombre', 'departamento', 'categoria', 'subcategoria',
             'marca', 'contenido', 'precio', 'descripcion']]

    print(f'{len(df):,} productos únicos')
    print(f'\nCobertura:')
    print(f'  Con marca    : {df["marca"].ne("").sum():,} ({df["marca"].ne("").mean()*100:.1f}%)')
    print(f'  Con contenido: {df["contenido"].notna().sum():,} ({df["contenido"].notna().mean()*100:.1f}%)')
    display(df.head(15))

    # guardar DataFrame raw
    ts      = pd.Timestamp.now().strftime('%Y%m%d_%H%M')
    archivo = f'jumbo_productos_{ts}.csv'    
    df.to_csv(f'../data/raw/{archivo}', index=False)  
 

    # Análisis de cobertura de contenido por categoría 
    print(f'\n{"="*50}')
    print('Cobertura de campo "contenido" por categoría:')

    df['contenido_vacio'] = (
        df['contenido'].isna() |
        df['contenido'].astype(str).str.strip().isin(['', 'None', 'nan'])
    )

    total   = df.groupby('categoria').size().reset_index(name='total')
    vacios  = (df[df['contenido_vacio']]
               .groupby('categoria').size()
               .reset_index(name='vacios'))

    resumen = (total
               .merge(vacios, on='categoria', how='left')
               .fillna(0)
               .assign(vacios=lambda x: x['vacios'].astype(int))
               .assign(pct_vacio=lambda x: (x['vacios'] / x['total'] * 100).round(1))
               .sort_values('pct_vacio', ascending=False)
               .reset_index(drop=True))

    display(resumen)

    # Filtrar categorías no alimentarias
    print(f'\n{"="*50}')
    print('Aplicando filtros de departamentos y categorías...')

    DEPTOS_DESCARTAR = [
        'Ropa Y Accesorios',
        'Especiales',
        'Mascotas',
        'Libros Y Papelería',
        'Juguetería',
    ]

    CATEGORIAS_DESCARTAR = [
        # Hogar
        'Organización de Ropa',
        'Muebles y Accesorios para Baño',
        'Terraza Y Exterior',
        # Herramientas y Ferretería
        'Cerrajería',
        'Grifería para Baño',
        'Grifería para Cocina',
        'Herramientas para Construcción',
        # Mundo Bebés
        'Desarrollo Y Estimulación',
        'Cuarto Del Bebé',
        'Lencería Y Decoración',
        'Paseo Y Viajes',
        # Deportes
        'Otros Deportes',
        'Bicicletas',
        # Pinturas
        'Herramientas Para Pintar',
        'Industrial',
    ]

    df_filtrado = df[
        ~df['departamento'].isin(DEPTOS_DESCARTAR) &
        ~df['categoria'].isin(CATEGORIAS_DESCARTAR) &
        (df['precio'] > 0)
    ].copy().reset_index(drop=True)

    print(f'  Registros originales : {len(df):,}')
    print(f'  Registros después    : {len(df_filtrado):,}')
    print(f'  Eliminados           : {len(df) - len(df_filtrado):,}')
    print(f'\nDepartamentos finales:\n{df_filtrado["departamento"].value_counts().to_string()}')

## Guardar DataFrame procesado

df_filtrado.to_csv('../data/processed/productos_retail_procesados.csv', index=False)


Total antes de limpiar: 28,449
27,776 productos únicos

Cobertura:
  Con marca    : 27,776 (100.0%)
  Con contenido: 15,831 (57.0%)


,nombre,departamento,categoria,subcategoria,marca,contenido,precio,descripcion
0,Olla Multicooker Black+Decker Pr100B 11 Funcio...,Electrodomésticos,Pequeños Electrodomésticos,Ollas Arroceras,BLACK+DECKER,5.7 Litros,317900.0,Olla Multicooker Black+Decker Pr100B 11 Funcio...
1,Multiolla Oster Turbo 14fun 5.7lt Diamondforce,Electrodomésticos,Pequeños Electrodomésticos,Ollas Arroceras,OSTER,5.7lt,499900.0,"Capacidad: 5.7 L, Funciones: 14 Funciones pred..."
2,Olla a Presión Eléctrica Universal Multi Pro 6...,Electrodomésticos,Pequeños Electrodomésticos,Ollas Arroceras,UNIVERSAL,6 Litros,429900.0,Olla a Presión Eléctrica Multi Pro Referencia:...
3,Olla Multifuncional IMUSA SPEEDY COOK 5 Litros,Electrodomésticos,Pequeños Electrodomésticos,Ollas Arroceras,IMUSA,5 Litros,486900.0,La Olla multifuncional Speedy Cook de IMUSA cu...
4,Olla a presion Home Elements 4.2L,Electrodomésticos,Pequeños Electrodomésticos,Ollas Arroceras,HOME ELEMENTS,"4,2 litros",124900.0,"Capacidad: 4,2 litros. Material: Aluminio anod..."
5,Olla Arrocera Home Elements HEAR04N 0.6 Litros...,Electrodomésticos,Pequeños Electrodomésticos,Ollas Arroceras,HOME ELEMENTS,0.6 Litros,109900.0,La porción perfecta empieza aquí OLLA ARROCERA...
6,Olla Fondue De Chocolate Para Derretir Facil Y...,Electrodomésticos,Pequeños Electrodomésticos,Ollas Arroceras,UNIMARC,None,93900.0,Disfruta de deliciosos postres con esta olla f...
7,Arrocera Multiusos Oster 7 Tazas Diamondforce,Electrodomésticos,Pequeños Electrodomésticos,Ollas Arroceras,OSTER,7 Tazas,199900.0,"Capacidad: 1.7 L Peso del Producto: 2,5 kg Vol..."
8,Arrocera Multiusos Oster 10 Tazas Diamondforce,Electrodomésticos,Pequeños Electrodomésticos,Ollas Arroceras,OSTER,10 Tazas,225900.0,"Capacidad: 1.8 L Peso del Producto: 2,8 kg Vol..."
9,Olla Arrocera Black+Decker 14 Tazas Función Au...,Electrodomésticos,Pequeños Electrodomésticos,Ollas Arroceras,BLACK+DECKER,14 Tazas,149900.0,Olla Arrocera Black+Decker 14 Tazas Función Au...



Cobertura de campo "contenido" por categoría:


,categoria,total,vacios,pct_vacio
0,Camaras y Drones,8,8,100.0
1,Ortopedia Y Fisioterapia,3,3,100.0
2,Smart Home,10,10,100.0
3,Lencería Y Decoración,6,6,100.0
4,Halloween,210,209,99.5
...,...,...,...,...
124,Motor Y Mantenimiento,9,0,0.0
125,Muebles y Accesorios para Baño,7,0,0.0
126,Pisos y Paredes Cerámicos,100,0,0.0
127,Pinturas Especiales,2,0,0.0



Aplicando filtros de departamentos y categorías...
  Registros originales : 27,776
  Registros después    : 20,434
  Eliminados           : 7,342

Departamentos finales:
departamento
Supermercado                 9567
Hogar Y Decoración           3506
Herramientas Y Ferretería    1821
Electrodomésticos            1179
Tecnología                    712
Mundo Bebés                   621
Pinturas                      610
Deportes Y Tiempo Libre       425
Televisores Y Audio           405
Jardín                        389
Pisos y Paredes               371
Automóvil                     251
Salud Y Bienestar             235
Celulares                     207
Baños                         106
Mundo Parrilla                 29
